# BERT Full Fine-Tuning: fixed hyperparameter search


In [1]:
%cd /content
!rm -rf LORA-course-project
!git clone https://github.com/NikitaBaranenkov/LORA-course-project.git
%cd LORA-course-project
!ls
!ls src

/content
Cloning into 'LORA-course-project'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 90 (delta 21), reused 68 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 685.60 KiB | 8.79 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/LORA-course-project
artifacts  data    notebooks  reports		src
configs    legacy  README.md  requirements.txt
constants.py  __init__.py  README.md


In [2]:
%pip uninstall -y torchao
%pip install -q --upgrade transformers accelerate datasets evaluate scikit-learn

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.6 MB/s eta 0:00:00


In [3]:
import gc
import inspect
import json
import random
import time
from itertools import product

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.constants import (
    BASE_MODEL_BERT,
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)

RUN_NAME = "bert_full_ft_hparam_search"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL_DIR = ARTIFACTS_DIR / "best_model"
RUN_TABLES_DIR = REPORTS_TABLES_DIR / "bert_full_ft"

SWEEP_RESULTS_PATH = RUN_TABLES_DIR / "bert_full_ft_hparam_search_validation_results.csv"
FINAL_TEST_RESULTS_PATH = RUN_TABLES_DIR / "bert_full_ft_best_hparam_test_results.csv"
EXPERIMENT_CONFIG_PATH = RUN_TABLES_DIR / "bert_full_ft_hparam_search_config.json"


## Импорты и настройка окружения

## Загрузка фиксированных разбиений данных

In [4]:
train_df = pd.read_csv(TRAIN_CSV_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_CSV_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

for split_df in (train_df, validation_df, test_df):
    split_df["label"] = split_df["label"].astype(int)

raw_dataframes = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

raw_datasets = DatasetDict({
    split_name: Dataset.from_pandas(split_df, preserve_index=False)
    for split_name, split_df in raw_dataframes.items()
})
raw_datasets = raw_datasets.rename_column("label", "labels")

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 8105
    })
    validation: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 1431
    })
    test: Dataset({
        features: ['text', 'labels', 'label_name'],
        num_rows: 2388
    })
})

## Токенизация текста

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_BERT)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"],
)
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

sample = tokenized_datasets["train"][0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8105 [00:00<?, ? examples/s]

Map:   0%|          | 0/1431 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: 1


## Метрики качества

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


def softmax_numpy(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    probabilities = np.exp(shifted_logits)
    return probabilities / np.sum(probabilities, axis=1, keepdims=True)


def main_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


assert SELECTION_METRIC == "macro_f1"
assert SELECTION_METRIC in compute_metrics((np.zeros((1, len(id2label))), np.array([0])))

## Фабрика моделей BERT full fine-tuning

Каждая конфигурация sweep получает заново инициализированную модель `bert-base-uncased` при одном и том же seed. Здесь не создаются adapters и не замораживается backbone: encoder и classification head обучаются совместно.


In [7]:
NUM_LABELS = len(id2label)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def new_full_ft_model():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_BERT,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )


set_global_seed(SEED)
print("Device:", device)
print("Base model:", BASE_MODEL_BERT)
print("Adaptation: full fine-tuning")


Device: cuda
Base model: bert-base-uncased
Adaptation: full fine-tuning


## Fixed grid для full fine-tuning

Для полного дообучения BERT критичны достаточно малый `learning_rate` и regularization. Поэтому проверяются `learning_rate` из {1e-5, 2e-5, 3e-5, 5e-5} и `weight_decay` из {0.0, 0.01}. В отличие от LoRA, learning rate уменьшен, поскольку обновляются все веса pretrained encoder.

`num_train_epochs=4`, `warmup_ratio=0.06`, train/eval batch sizes и early stopping сохранены как в BERT + LoRA. Сетка содержит 8 trial. Лучший trial выбирается исключительно по validation macro-F1.


In [8]:
FULL_FT_GRID = [
    {
        "learning_rate": learning_rate,
        "num_train_epochs": 4,
        "weight_decay": weight_decay,
        "warmup_ratio": 0.06,
    }
    for learning_rate, weight_decay in product([1e-5, 2e-5, 3e-5, 5e-5], [0.0, 0.01])
]

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LOGGING_STEPS = 50


def summarize_parameters(model, print_summary=False):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    frozen_names = [
        name for name, parameter in model.named_parameters() if not parameter.requires_grad
    ]
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    trainable_share = trainable_params / total_params

    if print_summary:
        print(f"Total parameters:     {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Trainable share:      {100 * trainable_share:.4f}%")

    assert hasattr(model, "classifier"), "Expected BERT classifier head was not found."
    assert all(parameter.requires_grad for parameter in model.classifier.parameters()), (
        "The BERT classification head must be trainable."
    )
    assert not frozen_names, f"Full fine-tuning found frozen parameters: {frozen_names[:10]}"
    assert trainable_share > 0.9999, "Full fine-tuning should train approximately 100% of parameters."
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_share": trainable_share,
    }


set_global_seed(SEED)
preview_model = new_full_ft_model()
preview_parameter_summary = summarize_parameters(preview_model, print_summary=True)
del preview_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Number of full fine-tuning configurations:", len(FULL_FT_GRID))
pd.DataFrame(FULL_FT_GRID)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     109,484,547
Trainable parameters: 109,484,547
Trainable share:      100.0000%
Number of full fine-tuning configurations: 8


,learning_rate,num_train_epochs,weight_decay,warmup_ratio
0,0.00001,4,0.00,0.06
1,0.00001,4,0.01,0.06
2,0.00002,4,0.00,0.06
3,0.00002,4,0.01,0.06
4,0.00003,4,0.00,0.06
5,0.00003,4,0.01,0.06
6,0.00005,4,0.00,0.06
7,0.00005,4,0.01,0.06


### Проверка forward pass до sweep


In [9]:
set_global_seed(SEED)
smoke_model = new_full_ft_model().to(device)
summarize_parameters(smoke_model, print_summary=True)
smoke_model.eval()

smoke_batch_size = min(2, len(tokenized_datasets["validation"]))
smoke_batch = tokenized_datasets["validation"][:smoke_batch_size]
smoke_inputs = {
    key: smoke_batch[key].to(device)
    for key in ("input_ids", "attention_mask")
}
with torch.no_grad():
    smoke_output = smoke_model(**smoke_inputs)

expected_logits_shape = (smoke_batch_size, NUM_LABELS)
assert tuple(smoke_output.logits.shape) == expected_logits_shape
print("Forward pass logits shape:", tuple(smoke_output.logits.shape))

del smoke_model, smoke_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     109,484,547
Trainable parameters: 109,484,547
Trainable share:      100.0000%
Forward pass logits shape: (2, 3)


## Hyperparameter search только по validation macro-F1

В этом блоке `test` не используется. Для каждого набора параметров `Trainer` сохраняет лучший epoch по validation macro-F1; затем среди восьми запусков выбирается checkpoint с наибольшим `val_macro_f1`.


In [10]:
assert SELECTION_METRIC == "macro_f1"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def make_training_arguments(config_id, config):
    training_args_kwargs = {
        "output_dir": str(CHECKPOINTS_DIR / f"config_{config_id:02d}"),
        "learning_rate": config["learning_rate"],
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "num_train_epochs": config["num_train_epochs"],
        "weight_decay": config["weight_decay"],
        "warmup_ratio": config["warmup_ratio"],
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": SELECTION_METRIC,
        "greater_is_better": True,
        "save_total_limit": 1,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
    }
    signature = inspect.signature(TrainingArguments.__init__)
    strategy_key = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
    training_args_kwargs[strategy_key] = "epoch"
    return TrainingArguments(**training_args_kwargs)


sweep_records = []
best_run = None
for config_id, config in enumerate(FULL_FT_GRID, start=1):
    print(f"\nTraining full fine-tuning configuration {config_id}/{len(FULL_FT_GRID)}:", config)
    set_global_seed(SEED)
    run_model = new_full_ft_model().to(device)
    parameter_summary = summarize_parameters(run_model)
    run_trainer = Trainer(
        model=run_model,
        args=make_training_arguments(config_id, config),
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    start_time = time.perf_counter()
    run_trainer.train()
    run_training_time_sec = time.perf_counter() - start_time
    validation_metrics = run_trainer.evaluate(
        tokenized_datasets["validation"],
        metric_key_prefix="validation",
    )
    assert run_trainer.state.best_model_checkpoint is not None

    record = {
        "config_id": config_id,
        **config,
        "val_macro_f1": float(validation_metrics["validation_macro_f1"]),
        "val_weighted_f1": float(validation_metrics["validation_weighted_f1"]),
        "val_accuracy": float(validation_metrics["validation_accuracy"]),
        "training_time_sec": run_training_time_sec,
        "best_checkpoint": run_trainer.state.best_model_checkpoint,
        **parameter_summary,
    }
    sweep_records.append(record)
    if best_run is None or record["val_macro_f1"] > best_run["record"]["val_macro_f1"]:
        best_run = {
            "record": dict(record),
            "config": dict(config),
            "parameter_summary": dict(parameter_summary),
        }

    del run_trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sweep_results_df = (
    pd.DataFrame(sweep_records)
    .sort_values("val_macro_f1", ascending=False)
    .reset_index(drop=True)
)
sweep_results_df.to_csv(SWEEP_RESULTS_PATH, index=False)

best_config = best_run["config"]
selected_training_time_sec = best_run["record"]["training_time_sec"]
print("\nBest full fine-tuning configuration selected only on validation macro-F1:")
print(best_config)
print("Best checkpoint:", best_run["record"]["best_checkpoint"])
sweep_results_df



Training full fine-tuning configuration 1/8: {'learning_rate': 1e-05, 'num_train_epochs': 4, 'weight_decay': 0.0, 'warmup_ratio': 0.06}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.426858,0.410866,0.842068,0.785367,0.838281
2,0.343894,0.405200,0.855346,0.796882,0.849433
3,0.220439,0.389541,0.865828,0.824909,0.865390
4,0.198641,0.404235,0.865129,0.825147,0.864505


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

[transformers] early stopping required metric_for_best_model, but did not find eval_macro_f1 so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.035733,0.548901,4,0.878407,0.839264,0.877091



Best full fine-tuning configuration selected only on validation macro-F1:
{'learning_rate': 2e-05, 'num_train_epochs': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.06}
Best checkpoint: /content/LORA-course-project/artifacts/bert_full_ft_hparam_search/checkpoints/config_04/checkpoint-1521


,config_id,learning_rate,num_train_epochs,weight_decay,warmup_ratio,val_macro_f1,val_weighted_f1,val_accuracy,training_time_sec,best_checkpoint,total_params,trainable_params,trainable_share
0,4,0.00002,4,0.01,0.06,0.841278,0.878952,0.879804,474.770186,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
1,3,0.00002,4,0.00,0.06,0.840162,0.877613,0.877708,331.984058,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
2,8,0.00005,4,0.01,0.06,0.839264,0.877091,0.878407,345.585252,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
3,7,0.00005,4,0.00,0.06,0.837118,0.873301,0.872816,405.409001,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
4,5,0.00003,4,0.00,0.06,0.836506,0.873496,0.872117,449.023481,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
5,6,0.00003,4,0.01,0.06,0.830391,0.869144,0.869322,422.371086,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
6,2,0.00001,4,0.01,0.06,0.829314,0.868591,0.868623,279.767234,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0
7,1,0.00001,4,0.00,0.06,0.825147,0.864505,0.865129,292.202614,/content/LORA-course-project/artifacts/bert_fu...,109484547,109484547,1.0


## Единственная финальная оценка на test split

Загружаем Checkpoint победившей validation-конфигурации заново.


In [11]:
assert FINAL_TEST_EVALUATION_ONLY is True
assert best_run is not None

set_global_seed(SEED)
best_model = AutoModelForSequenceClassification.from_pretrained(
    best_run["record"]["best_checkpoint"]
).to(device)

final_eval_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR / "final_evaluation"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    report_to="none",
    fp16=torch.cuda.is_available(),
)
best_trainer = Trainer(model=best_model, args=final_eval_args, compute_metrics=compute_metrics)
test_prediction_output = best_trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
test_logits = test_prediction_output.predictions
test_probabilities = softmax_numpy(test_logits)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = raw_dataframes["test"]["label"].to_numpy()
test_metrics = main_metrics(test_labels, test_predictions)

class_names = [id2label[class_id] for class_id in sorted(id2label)]
test_classification_report = classification_report(
    test_labels,
    test_predictions,
    labels=sorted(id2label),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
test_classification_report_df = (
    pd.DataFrame(test_classification_report).transpose().rename_axis("class").reset_index()
)
test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=sorted(id2label)),
    index=[f"true_{class_name}" for class_name in class_names],
    columns=[f"pred_{class_name}" for class_name in class_names],
)

test_predictions_df = raw_dataframes["test"].copy()
test_predictions_df["true_label"] = test_labels
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(id2label)
test_predictions_df["pred_label"] = test_predictions
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(id2label)
test_predictions_df["correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]
for class_id, class_name in id2label.items():
    test_predictions_df[f"prob_{class_name.lower()}"] = test_probabilities[:, class_id]

print("Final test metrics from the validation-selected full fine-tuning checkpoint:")
test_metrics


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final test metrics from the validation-selected full fine-tuning checkpoint:


{'accuracy': 0.8844221105527639,
 'macro_f1': 0.8499748745936843,
 'weighted_f1': 0.8841807576371349}

## Сохранение результатов и лучшей full fine-tuning модели

training_time_sec относится только к обучению выбранной validation-конфигурации, а не к длительности всего sweep


In [12]:
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULT_COLUMNS = [
    "model",
    "adaptation",
    "test_macro_f1",
    "test_weighted_f1",
    "test_accuracy",
    "bearish_f1",
    "bullish_f1",
    "neutral_f1",
    "total_params",
    "trainable_params",
    "trainable_share",
    "training_time_sec",
]

selected_parameter_summary = best_run["parameter_summary"]
final_test_row = {
    "model": BASE_MODEL_BERT,
    "adaptation": "full fine-tuning",
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "bearish_f1": float(test_classification_report["Bearish"]["f1-score"]),
    "bullish_f1": float(test_classification_report["Bullish"]["f1-score"]),
    "neutral_f1": float(test_classification_report["Neutral"]["f1-score"]),
    "total_params": selected_parameter_summary["total_params"],
    "trainable_params": selected_parameter_summary["trainable_params"],
    "trainable_share": selected_parameter_summary["trainable_share"],
    "training_time_sec": selected_training_time_sec,
}
final_test_results_df = pd.DataFrame([final_test_row], columns=FINAL_RESULT_COLUMNS)
assert list(final_test_results_df.columns) == FINAL_RESULT_COLUMNS
final_test_results_df.to_csv(FINAL_TEST_RESULTS_PATH, index=False)

test_predictions_df.to_csv(
    RUN_TABLES_DIR / "bert_full_ft_best_hparam_test_predictions.csv", index=False
)
test_classification_report_df.to_csv(
    RUN_TABLES_DIR / "bert_full_ft_best_hparam_test_classification_report.csv", index=False
)
test_confusion_matrix_df.to_csv(
    RUN_TABLES_DIR / "bert_full_ft_best_hparam_test_confusion_matrix.csv"
)

best_model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
experiment_config = {
    "dataset_name": DATASET_NAME,
    "model": BASE_MODEL_BERT,
    "adaptation": "full fine-tuning",
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "selection_metric": SELECTION_METRIC,
    "selection_split": "validation",
    "test_evaluation_after_validation_selection_only": FINAL_TEST_EVALUATION_ONLY,
    "fixed_grid": FULL_FT_GRID,
    "fixed_training_settings": {
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
    },
    "selected_config": best_config,
    "selected_best_checkpoint": best_run["record"]["best_checkpoint"],
    "selected_training_time_sec": selected_training_time_sec,
    "sweep_results_csv": str(SWEEP_RESULTS_PATH),
    "final_test_results_csv": str(FINAL_TEST_RESULTS_PATH),
}
with EXPERIMENT_CONFIG_PATH.open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Saved validation sweep table to:", SWEEP_RESULTS_PATH)
print("Saved final best-model test table to:", FINAL_TEST_RESULTS_PATH)
print("Saved validation-selected full fine-tuning model and tokenizer to:", BEST_MODEL_DIR)
final_test_results_df


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved validation sweep table to: /content/LORA-course-project/reports/tables/bert_full_ft/bert_full_ft_hparam_search_validation_results.csv
Saved final best-model test table to: /content/LORA-course-project/reports/tables/bert_full_ft/bert_full_ft_best_hparam_test_results.csv
Saved validation-selected full fine-tuning model and tokenizer to: /content/LORA-course-project/artifacts/bert_full_ft_hparam_search/best_model


,model,adaptation,test_macro_f1,test_weighted_f1,test_accuracy,bearish_f1,bullish_f1,neutral_f1,total_params,trainable_params,trainable_share,training_time_sec
0,bert-base-uncased,full fine-tuning,0.849975,0.884181,0.884422,0.809869,0.819915,0.92014,109484547,109484547,1.0,474.770186
